# 01 — Malliavin Calculus Primer

## What is the Malliavin derivative?

The **Malliavin derivative** D_s F is the Fréchet derivative of a random variable F
on Wiener space in the Cameron–Martin direction.  Informally, D_s F measures how
sensitive F is to a perturbation of the Brownian path at time s.

Under GBM  S_T = S₀ exp(μT + σW_T)  we have:

    D_s S_T = σ · S_T   for all s ≤ T

The derivative is **constant in s** — a perturbation at any time has the same
proportional effect on S_T.

## The integration-by-parts formula

The key result (Fournié et al., 1999):  for any measurable payoff f,

    ∂/∂S₀  E[f(S_T)]  =  E[ f(S_T) · π ]

where the **Malliavin weight**  π = W_T / (σ S₀ T)  is computed from the
simulated path — **without ever differentiating f**.

This works for discontinuous payoffs (digital options, barriers) because the
singularity of f′ is absorbed by the Gaussian measure via the integration by parts.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from mgreeks.models.gbm import GeometricBrownianMotion
from mgreeks.payoffs.european import EuropeanCall, DigitalCall
from mgreeks.simulation import MonteCarloEngine
from mgreeks.greeks import MalliavinGreeks, FiniteDifferenceGreeks
from mgreeks.greeks.analytical import bs_delta, bs_digital_delta
from mgreeks.weights.malliavin_weights import delta_weight_gbm

# Parameters
S0, K, T = 100.0, 100.0, 1.0
r, q, sigma = 0.05, 0.02, 0.20
n_paths, seed = 50_000, 42

model = GeometricBrownianMotion(r=r, q=q, sigma=sigma)
disc = np.exp(-r * T)

print(f"Model: GBM  r={r}  q={q}  σ={sigma}")
print(f"Option: K={K}  T={T}  S₀={S0}")


In [ ]:
## Step 1: Simulate paths and compute the weight by hand

rng = np.random.default_rng(seed)
out = model.simulate(S0, T, n_steps=1, n_paths=n_paths,
                     return_full_paths=True, rng=rng)

S_T = out["terminal"]             # terminal spot, shape (n_paths,)
W_T = out["brownian_increments"].sum(axis=1)   # Σ ΔWᵢ = W_T

# Delta weight formula: π = W_T / (σ S₀ T)
weight = delta_weight_gbm(S0, S_T, sigma, r, q, T, W_T)

print(f"W_T: mean={W_T.mean():.4f}  std={W_T.std():.4f}  (should be ~N(0,{T:.1f}))")
print(f"Weight π: mean={weight.mean():.4f}  std={weight.std():.4f}")


In [ ]:
## Step 2: European call — compute delta by hand

euro_payoff = EuropeanCall(K)
f = euro_payoff(out["paths"], out["times"])   # payoff on each path

samples = disc * f * weight                   # e^{-rT} f · π
delta_mall = samples.mean()
delta_se   = samples.std(ddof=1) / np.sqrt(n_paths)
delta_bs   = bs_delta(S0, K, T, r, q, sigma, "call")

print(f"Malliavin delta: {delta_mall:.5f} ± {delta_se:.5f}")
print(f"Black-Scholes:   {delta_bs:.5f}")
print(f"Error: {abs(delta_mall - delta_bs):.5f}  ({abs(delta_mall - delta_bs)/delta_se:.1f} SE)")


In [ ]:
## Step 3: Visualise the weight distribution

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("GBM Delta Weight  π = W_T / (σ S₀ T)", fontsize=13)

ax = axes[0]
ax.hist(W_T, bins=60, density=True, color="#1f77b4", alpha=0.7, label="W_T")
xs = np.linspace(-4, 4, 300)
ax.plot(xs, np.exp(-xs**2 / (2*T)) / np.sqrt(2*np.pi*T), "k--", lw=2, label=f"N(0,{T})")
ax.set_xlabel("W_T"); ax.set_ylabel("density"); ax.legend()
ax.set_title("Terminal Brownian  W_T ~ N(0,T)")

ax = axes[1]
weighted = f * weight
ax.hist(weighted[weighted != 0], bins=80, density=True, color="#d62728", alpha=0.7)
ax.axvline(weighted.mean(), color="k", lw=2, label=f"mean = {weighted.mean():.4f}")
ax.set_xlabel("f(S_T) · π"); ax.legend()
ax.set_title("Malliavin integrand  f * pi  (E[f*pi]*disc = delta)")

plt.tight_layout()
plt.savefig("01_weight_distribution.png", dpi=100, bbox_inches="tight")
plt.show()
print("Figure saved.")


In [ ]:
## Step 4: Payoff independence — apply the SAME weight to a digital call

# Digital call: f(S_T) = 1_{S_T > K}   (discontinuous!)
digital = DigitalCall(K)
f_dig = digital(out["paths"], out["times"])

samples_d = disc * f_dig * weight      # same weight π, different payoff
delta_dig_mall = samples_d.mean()
delta_dig_se   = samples_d.std(ddof=1) / np.sqrt(n_paths)
delta_dig_bs   = bs_digital_delta(S0, K, T, r, q, sigma)

# FD comparison (uses same paths internally, different bump)
engine = MonteCarloEngine(model, n_paths=n_paths, n_steps=1, rng_seed=seed)
fd = FiniteDifferenceGreeks(model, engine, bump_size=0.01, bump_type="relative")
delta_dig_fd   = fd.delta(digital, S0, T)

print("=" * 60)
print("Digital Call Delta  (f is discontinuous at K)")
print("=" * 60)
print(f"{'Method':<20}  {'estimate':>10}  {'SE':>10}  {'bias':>10}")
print("-" * 60)
print(f"{'Malliavin':<20}  {delta_dig_mall:>10.5f}  {delta_dig_se:>10.5f}  {delta_dig_mall - delta_dig_bs:>10.5f}")
print(f"{'FD (h=1%)':<20}  {delta_dig_fd['value']:>10.5f}  {delta_dig_fd['std_error']:>10.5f}  {delta_dig_fd['value'] - delta_dig_bs:>10.5f}")
print(f"{'Analytical':<20}  {delta_dig_bs:>10.5f}  {'—':>10}  {'—':>10}")
print()
print(f"FD SE / Malliavin SE = {delta_dig_fd['std_error'] / delta_dig_se:.1f}×  (Malliavin wins decisively)")
